# 🌾 Notebook 2: Wheat Plant Disease - Model Development & Fine-Tuning
**Dataset:** [`kushagra3204/wheat-plant-diseases`](https://www.kaggle.com/datasets/kushagra3204/wheat-plant-diseases)

In this notebook, we:
1. Install dependencies first.
2. Load and augment the wheat leaf images.
3. Build & train a clean **Baseline Custom CNN** model.
4. Build & **Fine-Tune a Pretrained CNN** (MobileNetV2) using Transfer Learning.
5. Save the trained models for performance analysis.

In [ ]:
# Step 1: Install all required dependencies
!pip install -q tensorflow numpy pandas matplotlib seaborn scikit-learn pillow
print('Dependencies installed successfully!')


In [ ]:
# Step 2: Import libraries
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print('TensorFlow version:', tf.__version__)
print('GPU Available:', tf.config.list_physical_devices('GPU'))


In [ ]:
# Step 3: Find dataset path automatically
def find_data_path():
    candidates = [
        '/kaggle/input/wheat-plant-diseases/data',
        '/kaggle/input/wheat-plant-diseases',
        '/kaggle/input/data',
        './data'
    ]
    for p in candidates:
        if os.path.exists(os.path.join(p, 'train')):
            return os.path.join(p, 'train'), os.path.join(p, 'valid')
    
    for root, dirs, files in os.walk('/kaggle/input' if os.path.exists('/kaggle/input') else '.'):
        if 'train' in dirs and 'valid' in dirs:
            return os.path.join(root, 'train'), os.path.join(root, 'valid')
    return None, None

TRAIN_DIR, VALID_DIR = find_data_path()
print('Train Path:', TRAIN_DIR)
print('Valid Path:', VALID_DIR)


In [ ]:
# Step 4: Data Augmentation & Generators
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Training generator with data augmentation (rotations, flips, zoom)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

# Validation generator (only rescaling, no distortion)
valid_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

valid_generator = valid_datagen.flow_from_directory(
    VALID_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

num_classes = len(train_generator.class_indices)
print(f'Total Classes: {num_classes}')


## 🏗️ Model 1: Baseline Custom CNN
A simple, clean Convolutional Neural Network built with Conv2D, MaxPooling, GlobalAveragePooling, and Dense layers.

In [ ]:
# Step 5: Build Baseline Custom CNN Model
baseline_model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    
    # Block 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D(2, 2),
    
    # Block 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D(2, 2),
    
    # Block 3
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D(2, 2),
    
    # Global Average Pooling (replaces heavy flatten layer to prevent overfitting)
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

baseline_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

baseline_model.summary()


In [ ]:
# Step 6: Train Baseline Model (10 Epochs)
print('Training Baseline Model...')
history_baseline = baseline_model.fit(
    train_generator,
    epochs=10,
    validation_data=valid_generator
)

# Save baseline model and history
baseline_model.save('baseline_model.h5')
with open('baseline_history.json', 'w') as f:
    json.dump({k: [float(x) for x in v] for k, v in history_baseline.history.items()}, f)
print('✅ Baseline model saved as baseline_model.h5')


## 🚀 Model 2: Fine-Tuning a Pretrained CNN (MobileNetV2)
To get significantly better accuracy, we use Transfer Learning:
1. **Phase 1 (Feature Extraction):** Keep MobileNetV2 base frozen and train only the top layer.
2. **Phase 2 (Fine-Tuning):** Unfreeze top layers of MobileNetV2 with a small learning rate (`1e-5`) so the model learns specific wheat disease patterns.

In [ ]:
# Step 7: Build Pretrained MobileNetV2 Model
# Load MobileNetV2 without the top layer
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# Phase 1: Freeze base model
base_model.trainable = False

# Add classification head on top
finetuned_model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    # Preprocessing for MobileNetV2 (scales pixel values between -1 and 1)
    layers.Lambda(lambda x: tf.keras.applications.mobilenet_v2.preprocess_input(x * 255.0)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

finetuned_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

finetuned_model.summary()


In [ ]:
# Step 8: Phase 1 Training - Train head for 5 epochs with frozen base
print('Starting Phase 1 Training (Feature Extraction)...')
history_p1 = finetuned_model.fit(
    train_generator,
    epochs=5,
    validation_data=valid_generator
)


In [ ]:
# Step 9: Phase 2 Training - Fine-Tune top 30 layers with low learning rate
print('Starting Phase 2 Deep Fine-Tuning...')
base_model.trainable = True

# Freeze all layers except the last 30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Recompile with a very small learning rate to avoid destroying weights
finetuned_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train for 7 more epochs
history_p2 = finetuned_model.fit(
    train_generator,
    epochs=12,
    initial_epoch=5,
    validation_data=valid_generator
)

# Save fine-tuned model and combined history
finetuned_model.save('finetuned_model.h5')

# Save combined history
combined_history = {}
for k in history_p1.history.keys():
    combined_history[k] = [float(x) for x in history_p1.history[k] + history_p2.history[k]]
with open('finetuned_history.json', 'w') as f:
    json.dump(combined_history, f)

# Save class indices mapping
with open('classes.json', 'w') as f:
    json.dump(valid_generator.class_indices, f, indent=4)

print('✅ Fine-tuned model saved as finetuned_model.h5')
print('✅ Class mappings saved as classes.json')


### ✅ Model Development Summary:
1. **Baseline CNN** trained for 10 epochs and saved as `baseline_model.h5`.
2. **Fine-Tuned MobileNetV2** trained in 2 phases (frozen feature extraction $\rightarrow$ fine-tuning top layers) and saved as `finetuned_model.h5`.
3. Both histories and class names are saved and ready for evaluation.

👉 Now open **`model_performance.ipynb`** to evaluate both models, compare accuracy, and plot confusion matrices!